什么是 TypedDict
TypedDict 是 Python 3.8+ 引入的一种类型提示工具，即带有类型声明的字典结构。适合需要快速定义
字典结构且无需 Pydantic 重量级功能的场景。
1、普通 dict 没有类型信息：

In [ ]:
{
    "title": "盗梦空间",
    "year": 2010,
    "director": "克里斯托弗·诺兰",
    "rating": 9.3
}

 TypedDict 可以进一步说明：
这个字典应该有哪些字段
每个字段的类型是什么
TypedDict 主要是类型声明，不是运行时强校验器

In [1]:
from typing_extensions import TypedDict


class MovieDict(TypedDict):
    title: str
    year: int
    director: str
    rating: float


#实例化字典时给出的字段名称和TypedDict不完全一致，此时IDE的静态类型检查会标记。但不会导致运行时异常，输出如下所示
movie: MovieDict = {
    "title1": "盗梦空间",
    "year": 2010,
    "director": "克里斯托弗·诺兰",
    "rating": 8.8,
}
print(movie)

{'title1': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰', 'rating': 8.8}


2.2.2 基本使用
Annotated的使用
Annotated 用来在“类型”之外，再附加一些额外信息，即元数据。类似于Pydantic的Field。

In [3]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
model_with_closeai = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}}  #关闭思考模式
)

举例1：返回简单结构

In [10]:
"""
使用 TypedDict 模型定义结构化输出
"""
from typing_extensions import TypedDict, Annotated


class MovieTypedDict(TypedDict):
    """
    电影的详细信息
    """
    title: Annotated[str, "电影的正式名称，例如《盗梦空间》"]
    year: Annotated[int, "电影的公映年份，使用四位数字表示"]
    director: Annotated[str, "电影导演的全名"]
    rating: Annotated[float, "电影在10分制下的评分，可包含一位小数"]


# 设置模型结构化输出
structured_llm = model_with_closeai.with_structured_output(MovieTypedDict, method="json_schema")
print(structured_llm)
# 调用模型并获取结构化输出
response = structured_llm.invoke("给我介绍下电影《星际穿越》")
print(type(response))
print(response)

first=RunnableBinding(bound=ChatDeepSeek(profile={}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000028902DD7770>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000289036B42F0>, root_client=<openai.OpenAI object at 0x0000028902C62660>, root_async_client=<openai.AsyncOpenAI object at 0x0000028902DD78C0>, model_name='deepseek-v4-flash', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.deepseek.com', extra_body={'thinking': {'type': 'disabled'}}, api_key=SecretStr('**********'), api_base='https://api.deepseek.com/v1'), kwargs={'tools': [{'type': 'function', 'function': {'name': 'MovieTypedDict', 'description': '电影的详细信息', 'parameters': {'type': 'object', 'properties': {'title': {'default': '电影的正式名称，例如《盗梦空间》', 'type': 'string'}, 'year': {'default': '电影的公映年份，使用四位数字表示', 'type': 'integer'}, 'director': {'default': '电影导演的全名', 'type': 'string'}, 'rating': {'default': '电影在10分制下的评分，

举例2：返回嵌套结构

In [11]:
from typing import TypedDict, List, Annotated


# 使用TypedDict定义嵌套结构
class Actor(TypedDict):
    """演员情况"""
    name: Annotated[str, "演员姓名"]
    role: Annotated[str, "饰演的角色"]


class Movie(TypedDict):
    """电影情况"""
    title: Annotated[str, "电影标题"]
    year: Annotated[int, "上映年份"]
    director: Annotated[str, "导演"]
    cast: Annotated[List[Actor], "演员列表"]  # 嵌套列表定义
    rating: Annotated[float, "评分"]


# 设置模型结构化输出
structured_llm = model_with_closeai.with_structured_output(Movie, method="json_schema")
print(structured_llm)
# 调用模型并获取结构化输出
resp = structured_llm.invoke("给我介绍下电影《盗梦空间》")
print(resp)

first=RunnableBinding(bound=ChatDeepSeek(profile={}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000028902DD7770>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000289036B42F0>, root_client=<openai.OpenAI object at 0x0000028902C62660>, root_async_client=<openai.AsyncOpenAI object at 0x0000028902DD78C0>, model_name='deepseek-v4-flash', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.deepseek.com', extra_body={'thinking': {'type': 'disabled'}}, api_key=SecretStr('**********'), api_base='https://api.deepseek.com/v1'), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '电影情况', 'parameters': {'type': 'object', 'properties': {'title': {'default': '电影标题', 'type': 'string'}, 'year': {'default': '上映年份', 'type': 'integer'}, 'director': {'default': '导演', 'type': 'string'}, 'cast': {'default': '演员列表', 'type': 'array', 'items': {'description': '演员情况'

举例3：...的使用
说明：与模型提供商有关系
DEEPSEEK模型

In [13]:
from typing_extensions import TypedDict, Annotated


class MovieDict(TypedDict):
    """
    电影的详细信息
    """
    title: Annotated[str, ..., "电影标题"]
    year: Annotated[int, ..., "电影上映年份"]
    director: Annotated[str, ..., "导演"]
    rating: Annotated[float, ..., "电影评分，满分十分"]


model_with_structure = model_with_closeai.with_structured_output(MovieDict)
print(model_with_structure)
response = model_with_structure.invoke("根据这段话抽取盗梦空间的信息，不包含的信息可以留空：盗梦空间在2010年上映，导演是克里斯托弗·诺兰。")
print(response)
print(type(response))

first=RunnableBinding(bound=ChatDeepSeek(profile={}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000028902DD7770>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000289036B42F0>, root_client=<openai.OpenAI object at 0x0000028902C62660>, root_async_client=<openai.AsyncOpenAI object at 0x0000028902DD78C0>, model_name='deepseek-v4-flash', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.deepseek.com', extra_body={'thinking': {'type': 'disabled'}}, api_key=SecretStr('**********'), api_base='https://api.deepseek.com/v1'), kwargs={'tools': [{'type': 'function', 'function': {'name': 'MovieDict', 'description': '电影的详细信息', 'parameters': {'type': 'object', 'properties': {'title': {'description': '电影标题', 'type': 'string'}, 'year': {'description': '电影上映年份', 'type': 'integer'}, 'director': {'description': '导演', 'type': 'string'}, 'rating': {'description': '电影评分，满分十分', 'type': 'number'